# Silver → Gold 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## 2. Dimensão Motivo
### Regra de negócio e estrutura
A dimensão de motivos disponibiliza os atributos usados para classificar os motivos de atendimento:
- id_motivo: identificador do motivo e chave da dimensão;
- nome_motivo: descrição do motivo;
- categoria: categoria padronizada na Silver;
- criticidade: nível de criticidade padronizado na Silver.
A etapa Silver já converte o identificador para inteiro, remove espaços extras, trata categorias ausentes conhecidas e padroniza acentuação e criticidade.

In [0]:
df_silver_motivos = spark.table(f"{silver_schema}.tb_motivos")

df_gold_motivo = (
    df_silver_motivos
    .select(
        F.col("id_motivo"),
        F.col("nome_motivo"),
        F.col("categoria"),
        F.col("criticidade")
    )
)
(
    df_gold_motivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_motivo")
)

print(f"Tabela {gold_schema}.dim_motivo salva com sucesso!")
print(f"Total de motivos na dimensão: {df_gold_motivo.count()}")

display(df_gold_motivo.orderBy("id_motivo"))

Tabela vcredit.gold.dim_motivo salva com sucesso!
Total de motivos na dimensão: 14


id_motivo,nome_motivo,categoria,criticidade
3,Contestação de fatura,Financeiro,Alta
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,Baixa
5,Consulta de limite,Financeiro,Baixa
6,Desbloqueio de cartão,Cartão,Alta
7,Compra não autorizada,Cartão,Alta
8,Consulta de contrato,Financeiro,Média
9,Problema com aplicativo,Atendimento,Média
10,Consulta de fatura,Financeiro,Baixa
11,Bloqueio de cartão,Cartão,Alta
12,Contratação de cartão adicional,Cartão,Média


## Dimensão Data
## 03 · Silver → Gold: Modelagem Dimensional (Dimensão Data)

A tabela `vcredit.gold.dim_data` é a dimensão de tempo do modelo, com um registro para cada dia e atributos de calendário que permitem filtrar e agrupar os chamados por dia, mês, ano e dia da semana. Sua estrutura segue a **Modelagem Dimensional (*Star Schema*)**.

### Objetivo desta etapa:
- Gerar um calendário com um registro para cada dia, abrangendo todos os anos em que há chamados registrados.
- Disponibilizar atributos de calendário pré-calculados (componentes da data, trimestre, dia da semana e indicador de fim de semana), eliminando a necessidade de calculá-los nos relatórios.
- Garantir a chave `id_tempo` como elo de ligação entre a dimensão e a tabela fato.
- Assegurar idempotência na gravação em formato **Delta Lake**, permitindo a reexecução sem duplicidade de registros.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

catalog = "vcredit"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Lendo da Silver: {silver_schema}")
print(f"Gravando na Gold: {gold_schema}")

Lendo da Silver: vcredit.silver
Gravando na Gold: vcredit.gold


### 1. Dimensão Data (`vcredit.gold.dim_data`)

### Regra de Negócio e Enriquecimento:
O calendário não é originado de uma tabela de origem, sendo gerado a partir do intervalo de anos identificado nos chamados da camada Silver.
- **Intervalo de cobertura**: de 01/jan do ano do chamado mais antigo a 31/dez do ano do chamado mais recente, com base em `silver.tb_chamados_hora`, desconsiderando os registros sinalizados em `dq_fora_periodo` ou `dq_abertura_nula`;
- **`id_tempo`**: chave primária (`PK`) no formato `yyyyMMdd` (ex.: `20260525`). A tabela fato gera o mesmo valor a partir da data de abertura do chamado, estabelecendo o relacionamento entre as tabelas (`FK`);
- **`data`**, **`dia`**, **`mes`** e **`ano`**: data completa e seus respectivos componentes;
- **`dia_semana`**: valor de 1 (segunda-feira) a 7 (domingo), utilizado para ordenação nos gráficos;
- **`nome_dia_semana`**: denominação do dia da semana por extenso, destinada à exibição;
- **`trimestre`** e **`fim_de_semana`**: atributos de apoio a filtros e comparações;
- **Reprocessamento**: a tabela é integralmente regravada (`overwrite`) a cada execução, o que impede a duplicidade de registros em caso de reexecução.

In [0]:
df_silver_chamados_hora = spark.table(f"{silver_schema}.tb_chamados_hora")

limites = (
    df_silver_chamados_hora
    .filter(~F.col("dq_fora_periodo") & ~F.col("dq_abertura_nula"))
    .agg(F.min("data_abertura").alias("data_min"), F.max("data_abertura").alias("data_max"))
    .collect()[0]
)

data_inicio = f"{limites['data_min'].year}-01-01"
data_fim = f"{limites['data_max'].year}-12-31"

nomes_dia = ["Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira",
             "Sexta-feira", "Sábado", "Domingo"]

df_datas = spark.sql(
    f"SELECT explode(sequence(to_date('{data_inicio}'), to_date('{data_fim}'), interval 1 day)) AS data"
)

# dayofweek do Spark inicia no domingo (1); o valor é reajustado para segunda = 1 ... domingo = 7
dia_semana = ((F.dayofweek("data") + 5) % 7) + 1

df_gold_data = (
    df_datas
    .select(
        F.date_format("data", "yyyyMMdd").cast(IntegerType()).alias("id_tempo"),
        F.col("data"),
        F.dayofmonth("data").alias("dia"),
        F.month("data").alias("mes"),
        F.year("data").alias("ano"),
        dia_semana.alias("dia_semana"),
        F.element_at(F.array(*[F.lit(n) for n in nomes_dia]), dia_semana).alias("nome_dia_semana"),
        F.quarter("data").alias("trimestre"),
        (dia_semana >= 6).alias("fim_de_semana")
    )
)

(
    df_gold_data.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_data")
)

total_dias = df_gold_data.count()
total_chaves = df_gold_data.select("id_tempo").distinct().count()

print(f"Tabela {gold_schema}.dim_data salva com sucesso!")
print(f"Total de dias na dimensão: {total_dias} (de {data_inicio} até {data_fim})")
print(f"Chave id_tempo única: {total_dias == total_chaves}")
display(df_gold_data.orderBy("id_tempo"))


Tabela vcredit.gold.dim_data salva com sucesso!
Total de dias na dimensão: 730 (de 2025-01-01 até 2026-12-31)
Chave id_tempo única: True


id_tempo,data,dia,mes,ano,dia_semana,nome_dia_semana,trimestre,fim_de_semana
20250101,2025-01-01,1,1,2025,3,Quarta-feira,1,false
20250102,2025-01-02,2,1,2025,4,Quinta-feira,1,false
20250103,2025-01-03,3,1,2025,5,Sexta-feira,1,false
20250104,2025-01-04,4,1,2025,6,Sábado,1,true
20250105,2025-01-05,5,1,2025,7,Domingo,1,true
20250106,2025-01-06,6,1,2025,1,Segunda-feira,1,false
20250107,2025-01-07,7,1,2025,2,Terça-feira,1,false
20250108,2025-01-08,8,1,2025,3,Quarta-feira,1,false
20250109,2025-01-09,9,1,2025,4,Quinta-feira,1,false
20250110,2025-01-10,10,1,2025,5,Sexta-feira,1,false


## 3. Dimensão Canal
###Regra de negócio e estrutura
A dimensão de canais disponibiliza os atributos de identificação e classificação dos canais:
- id_canal: identificador do canal e chave da dimensão;
- nome_canal: nome do canal;
- tipo_canal: tipo ou classificação do canal;
- status_canal: status cadastral do canal.
A dimensão utiliza a tabela de canais já tratada na Silver. A seleção dos atributos mantém o escopo definido para a dimensão e evita repetir nesta etapa as transformações de limpeza realizadas anteriormente.

In [0]:
df_silver_canais = spark.table(f"{silver_schema}.canais")

df_gold_canal = (
    df_silver_canais
    .select(
        F.col("id_canal"),
        F.col("nome_canal"),
        F.col("tipo_canal"),
        F.col("status_canal")
    )
)
(
    df_gold_canal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_canal")
)

print(f"Tabela {gold_schema}.dim_canal salva com sucesso!")
print(f"Total de canais na dimensão: {df_gold_canal.count()}")

display(df_gold_canal.orderBy("id_canal"))

Tabela vcredit.gold.dim_canal salva com sucesso!
Total de canais na dimensão: 4


id_canal,nome_canal,tipo_canal,status_canal
1,URA,autosservico,ativo
2,Chatbot,autosservico,ativo
3,Atendimento Inicial,humano,ativo
4,Atendimento Especializado,humano,ativo


## 03 · Silver → Gold: Modelagem Dimensional (Dimensões de Atendimento)

Neste notebook construímos as tabelas da camada **Gold** sob a ótica da **Modelagem Dimensional (*Star Schema*)** para o Case V-Credit.

### Objetivo desta etapa:
- Consumir as tabelas limpas e padronizadas da camada Silver (`vcredit.silver.tb_atendentes` e `vcredit.silver.tb_motivos`).
- Aplicar regras analíticas de negócio, como o cálculo de **`tempo_empresa` (em dias)**.
- Disponibilizar as dimensões com nomenclatura de consumo analítico (`dim_atendente` e `dim_motivo`) prontas para relacionamento direto com a tabela fato no **Power BI**.
- Garantir idempotência e integridade na gravação em formato **Delta Lake**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

catalog = "vcredit"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Lendo da Silver: {silver_schema}")
print(f"Gravando na Gold: {gold_schema}")

Lendo da Silver: vcredit.silver
Gravando na Gold: vcredit.gold


---
## 1. Dimensão Atendente (`vcredit.gold.dim_atendente`)

### Regra de Negócio e Enriquecimento:
A dimensão de atendente requer os atributos cadastrais e a métrica de tempo de permanência na empresa:
- **`id_atendente`**: Chave primária da dimensão (`PK`);
- **`nome_atendente`**, **`nivel_atendimento`**, **`status_atendente`**;
- **`data_admissao`** e **`data_desligamento`**;
- **`tempo_empresa` (em dias)**: calculado dinamicamente:
  - Se o atendente foi desligado (`data_desligamento IS NOT NULL`): `datediff(data_desligamento, data_admissao)`.
  - Se o atendente continua ativo (`data_desligamento IS NULL`): `datediff(current_date(), data_admissao)`.

In [0]:
df_silver_atendentes = spark.table(f"{silver_schema}.tb_atendentes")

df_gold_atendente = (
    df_silver_atendentes
    .withColumn(
        "tempo_empresa",
        F.when(
            F.col("data_desligamento").isNotNull(),
            F.datediff(F.col("data_desligamento"), F.col("data_admissao"))
        ).otherwise(
            F.datediff(F.current_date(), F.col("data_admissao"))
        )
    )
    .select(
        F.col("id_atendente"),
        F.col("nome_atendente"),
        F.col("nivel_atendimento"),
        F.col("data_admissao"),
        F.col("data_desligamento"),
        F.col("status_atendente"),
        F.col("tempo_empresa").cast(IntegerType()).alias("tempo_empresa")
    )
)

df_silver_chamados = spark.table(f"{silver_schema}.tb_chamados")

df_atendentes_orfaos = (
    df_silver_chamados
    .select("id_atendente")
    .filter(F.col("id_atendente").isNotNull())
    .distinct()
    .join(
        df_gold_atendente.select("id_atendente"),
        on="id_atendente",
        how="left_anti"
    )
)

df_atendentes_orfaos = (
    df_atendentes_orfaos
    .select(
        F.col("id_atendente"),
        F.lit("Desconhecido").alias("nome_atendente"),
        F.lit("Desconhecido").alias("nivel_atendimento"),
        F.lit(None).cast("date").alias("data_admissao"),
        F.lit(None).cast("date").alias("data_desligamento"),
        F.lit("Desconhecido").alias("status_atendente"),
        F.lit(None).cast("int").alias("tempo_empresa")
    )
)

df_gold_atendente = df_gold_atendente.unionByName(df_atendentes_orfaos)

print(f"Atendentes órfãos incluídos: {df_atendentes_orfaos.count()}")

(
    df_gold_atendente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_atendente")
)

print(f"Tabela {gold_schema}.dim_atendente salva com sucesso!")
print(f"Total de atendentes na dimensão: {df_gold_atendente.count()}")
display(df_gold_atendente)


Atendentes órfãos incluídos: 3
Tabela vcredit.gold.dim_atendente salva com sucesso!
Total de atendentes na dimensão: 28


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,tempo_empresa
1,Marcela França,Nível 1,2019-04-18,2025-11-16,Inativo,2404
10,Marcos Torres,Nível 1,2020-01-20,null,Ativo,2473
11,Maria Eduarda Freitas,Nível 1,2019-01-30,null,Ativo,2828
12,Marilene Nascimento,Nível 1,2021-11-25,null,Ativo,1798
13,Daniel Fonseca,Nível 1,2019-06-19,null,Ativo,2688
14,Patrícia Barbosa,Nível 2,2019-08-24,null,Ativo,2622
15,Josefa Araújo,Nível 2,2018-12-26,null,Ativo,2863
16,Eduardo Paiva,Nível 2,2022-04-28,null,Ativo,1644
17,Roberta Garcia,Nível 2,2021-12-02,null,Ativo,1791
18,Cristiane Loureiro,Nível 2,2019-08-27,null,Ativo,2619


### Dimensão Cliente (Gold)

Nesta etapa foi criada a dimensão `dim_cliente`, responsável por centralizar as principais informações cadastrais dos clientes para análises no modelo dimensional.

Também foram criados atributos derivados, como `faixa_etaria` e `tempo_base_cliente`, que facilitam análises segmentadas sem necessidade de novos cálculos durante o consumo dos dados.

In [0]:
df_clientes = spark.table(
    f"{silver_schema}.tb_clientes"
)

df_chamados = spark.table(
    f"{silver_schema}.tb_chamados"
)

clientes_faltantes = (
    df_chamados
    .select("id_cliente")
    .distinct()
    .join(
        df_clientes.select("id_cliente"),
        "id_cliente",
        "left_anti"
    )
)

clientes_faltantes = (
    clientes_faltantes
    .withColumn("nome", F.lit("DESCONHECIDO"))
    .withColumn("email", F.lit("DESCONHECIDO"))
    .withColumn("regiao", F.lit("DESCONHECIDO"))
    .withColumn("idade", F.lit(None).cast("int"))
    .withColumn("data_cadastro", F.lit(None).cast("date"))
    .withColumn("ano_cadastro", F.lit(None).cast("int"))
    .withColumn("dia_vencimento_fatura", F.lit(None).cast("int"))
)

df_clientes = df_clientes.unionByName(
    clientes_faltantes,
    allowMissingColumns=True
)

dim_cliente = (
    df_clientes

    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade") < 29, "Jovem")
         .when(F.col("idade") < 59, "Adulto")
         .when(F.col("idade").isNull(), "Não Informado")
         .otherwise("Idoso")
    )

    .withColumn(
        "tempo_base_cliente",
        F.round(
            F.months_between(
                F.current_date(),
                F.col("data_cadastro")
            ) / 12,
            1
        )
    )

    .select(
        "id_cliente",
        "nome",
        "email",
        "regiao",
        "idade",
        "faixa_etaria",
        "data_cadastro",
        "tempo_base_cliente",
        "dia_vencimento_fatura"
    )
)

dim_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_cliente"
    )

display(
    dim_cliente.filter(
        F.col("nome") == "DESCONHECIDO"
    )
)

id_cliente,nome,email,regiao,idade,faixa_etaria,data_cadastro,tempo_base_cliente,dia_vencimento_fatura
21229760351,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
81013942838,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
71554925965,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
40783772188,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
40180625242,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
86183361264,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
50029801441,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
57942262990,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
75635014698,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null
60598048779,DESCONHECIDO,DESCONHECIDO,DESCONHECIDO,null,Não Informado,null,null,null


## fact_chamados

In [0]:
df_pesquisa_satisfacao = (
    spark.table(f"{silver_schema}.pesquisa_satisfacao")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_custo = (
    spark.table(f"{silver_schema}.custos")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_hora = (
    spark.table(f"{silver_schema}.tb_chamados_hora")
    .select(
        F.col("id_chamado").cast("string").alias("id_chamado"),
        F.date_format("data_abertura", "yyyyMMdd")
         .cast("int").alias("id_data"),

        F.when(
            F.col("espera_valida"),
            F.col("espera_seg")
        ).cast("decimal(18,2)").alias("tempo_espera_seg"),

        F.when(
            F.col("atendimento_valido"),
            F.col("atendimento_seg")
        ).cast("decimal(18,2)").alias("tempo_atendimento_seg")
    )
)

df_fact = (
    df_chamados
    .join(df_pesquisa_satisfacao, on="id_chamado", how="left")
    .join(df_custo, on="id_chamado", how="left")
    .join(df_gold_motivo.withColumnRenamed("nome_motivo","motivo"), on="motivo", how="left")
    .join(df_gold_canal.withColumnRenamed("nome_canal","canal"), on="canal", how="left")
    .join(df_hora, on="id_chamado", how="left")
    .select(
        "id_chamado",
        F.col("id_cliente").cast("string").alias("id_cliente"),
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.col("id_motivo").cast("int").alias("id_motivo"),
        F.col("id_canal").cast("string").alias("id_canal"),
        F.col("id_data").cast("int").alias("id_data"),
        F.col("resolvido").cast("string").alias("resolvido"),
        F.when(F.col("custo_valido"), F.col("custo")).cast("decimal(18,2)").alias("custo"),
        F.when(F.col("nota_valida"), F.col("nota_atendimento")).cast("int").alias("nota_atendimento"),
        F.col("tempo_espera_seg"),
        F.col("tempo_atendimento_seg")
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_chamados")

print(f"fact_chamados: {df_fact.count()} chamados")
display(df_fact.limit(10))

fact_chamados: 131838 chamados


id_chamado,id_cliente,id_atendente,id_motivo,id_canal,id_data,resolvido,custo,nota_atendimento,tempo_espera_seg,tempo_atendimento_seg
45000,21748174497,20,15,2,20250930,Não Informado,0.60,4,0.00,106.00
118384,98181934335,13,3,3,20260624,Não,22.17,2,1155.00,1825.00
47744,33407772125,9,9,2,20250912,Sim,null,3,0.00,23.00
73423,58562080895,9,5,3,20260102,Sim,18.32,4,530.00,1583.00
113715,62506762413,2,11,2,20260624,Sim,0.87,null,0.00,176.00
52850,48960047270,11,16,3,20251014,Não,12.72,3,69.00,748.00
53117,85886180125,17,6,4,20251014,Sim,39.29,2,887.00,1319.00
82642,96596071759,6,9,3,20260215,Sim,18.44,5,362.00,1568.00
129978,93115110685,16,3,4,20260707,Sim,47.82,3,845.00,2099.00
30918,25258325654,23,16,2,20250701,Não,0.45,5,0.00,89.00
